In [1]:
from migration import ddl_resolver
from migration.ddl_resolver import DdlResolver
from src.utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer, CurDecomposerWriter
from migration.cur.metadata import CurMetadataProcessor
from migration.cur.generator import CurPySparkGenerator
from src.utils.source_rule_loader import load_all_source_rules

from src.paths import *

In [2]:
print(USERNAME)


output_root = PROJECT_ROOT / "output" / "migration"
all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()["cur"]

ext_giadung


In [3]:

# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_broker",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_address",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_crs",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_einvoice_customer_address",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile",
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment",


def get_paths(script_name):
    input_file = DATALAKE_SCRIPT_DIR /"dml" / "unl" /  f"unl_{script_name}.sql"

    file_name = os.path.basename(input_file).replace('.sql', '')
    layer, sub_layer, source_name, base_table = parse_file_name(input_file)

    print(f"File gốc tại: {input_file}")

    return input_file, file_name, input_file

In [4]:
import re
import sqlglot
from sqlglot import exp

def sql_to_pyspark_df(sql: str) -> str:
    # 1. Temporarily replace ${var} to avoid sqlglot parsing them as syntax errors
    safe_sql = re.sub(r'\$\{([^}]+)\}', r'__VAR_\1__', sql)

    # 2. Parse the SQL
    ast = sqlglot.parse_one(safe_sql, read="hive")

    # Extract the SELECT statement (ignoring INSERT OVERWRITE DIRECTORY)
    select_ast = ast.expression if isinstance(ast, exp.Insert) else ast

    def walk(node):
        """Recursively translates SQLGlot AST nodes to PySpark syntax."""
        if isinstance(node, exp.Column):
            # Drop table aliases like T.etl_dt -> etl_dt for cleaner PySpark
            return f'col("{node.name}")'

        elif isinstance(node, exp.Alias):
            alias_name = node.alias
            # If selecting a column and aliasing it to the exact same name, skip the alias
            if isinstance(node.this, exp.Column) and node.this.name.upper() == alias_name.upper():
                 return walk(node.this)

            # If selecting a literal value (like the ETL_TIMESTAMP parameter)
            if isinstance(node.this, exp.Literal):
                val = node.this.name
                if val.startswith("__VAR_") and val.endswith("__"):
                    var_name = val[6:-2]
                    return f'lit(params["{var_name}"]).alias("{alias_name}")'
                return f'lit("{val}").alias("{alias_name}")'

            return f'{walk(node.this)}.alias("{alias_name}")'

        elif isinstance(node, (exp.Anonymous, exp.Func)):
            func_name = node.name.upper()
            if func_name == 'REPLACE':
                args = list(node.expressions)
                this = walk(args[0])
                search = walk(args[1])
                replace = walk(args[2])
                return f'regexp_replace({this}, {search}, {replace})'
            elif func_name == 'CHR':
                val = node.expressions[0].name
                if val == '13': return '"\\r"'
                if val == '10': return '"\\n"'
                return f'chr({val})'
            return f'{func_name.lower()}({", ".join(walk(e) for e in node.expressions)})'

        elif isinstance(node, exp.Literal):
            val = node.name
            if val.startswith("__VAR_") and val.endswith("__"):
                var_name = val[6:-2]
                return f'params["{var_name}"]'
            if node.is_string:
                return f'"{val}"'
            return str(val)

        elif isinstance(node, exp.In):
            this = walk(node.this)
            exprs = [walk(e) for e in node.expressions]
            return f'{this}.isin([{", ".join(exprs)}])'

        elif isinstance(node, exp.EQ):
            return f'{walk(node.left)} == {walk(node.right)}'

        elif isinstance(node, exp.And):
            return f'{walk(node.left)} & {walk(node.right)}'

        return f'col("{node.name}")' # Fallback

    # --- Extract FROM ---
    table_node = select_ast.find(exp.Table)
    db = table_node.db
    name = table_node.name

    if db.startswith("__VAR_") and db.endswith("__"):
        db = f'{{params["{db[6:-2]}"]}}'

    table_str = f'f"""{db}.{name}"""' if db else f'"{name}"'

    # --- Extract WHERE ---
    filters = []
    if select_ast.args.get("where"):
        where_expr = select_ast.args["where"].this

        # Flatten AND conditions into separate .filter() calls
        def extract_ands(expr):
            if isinstance(expr, exp.And):
                return extract_ands(expr.left) + extract_ands(expr.right)
            return [expr]

        for cond in extract_ands(where_expr):
            filters.append(walk(cond))

    # --- Extract SELECT ---
    selects = [walk(proj) for proj in select_ast.expressions]

    # --- Code Assembly ---
    lines = [f"df = (", f'    spark.table({table_str})']

    for f in filters:
        lines.append(f'    .filter({f})')

    lines.append('    .select(')
    for i, s in enumerate(selects):
        prefix = "        " if i == 0 else "        ,"
        lines.append(f'{prefix}{s}')
    lines.append('    )')
    lines.append(')')

    return "\n".join(lines)


# Test with your SQL
sql_query = """
insert overwrite directory '/data/output/einv/${batch_date}/cur/cur_einvoice_account_f.${batch_date}.dat'
row format delimited
fields terminated by '\001'
lines terminated by '\n'
null defined as ''
select
     replace(replace(account_id,chr(13),''),chr(10),'') as account_id
    ,replace(replace(customer_id	,chr(13),''),chr(10),'') as customer_id
    ,replace(replace(agent_id	,chr(13),''),chr(10),'') as agent_id
    ,replace(replace(status_code,chr(13),''),chr(10),'') as status_code
    ,replace(replace(cust_full_name	,chr(13),''),chr(10),'') as cust_full_name
    ,replace(replace(home_branch	,chr(13),''),chr(10),'') as home_branch
    ,replace(replace(supplier_id,chr(13),''),chr(10),'') as supplier_id
    ,replace(replace(source_system_id	,chr(13),''),chr(10),'') as source_system_id
    ,replace(replace(lob_type	,chr(13),''),chr(10),'') as lob_type
    ,replace(replace(islamic_flag		,chr(13),''),chr(10),'') as islamic_flag
    ,replace(replace(account_type,chr(13),''),chr(10),'') as account_type
    ,replace(replace(client_group,chr(13),''),chr(10),'') as client_group
    ,replace(replace(account_classification,chr(13),''),chr(10),'') as account_classification
    ,replace(replace(tin_number,chr(13),''),chr(10),'') as tin_number
    ,'${batch_timestamp}' AS etl_timestamp
    ,etl_dt
from ${cur_schema}.einvoice_account t
where t.etl_dt = '${batch_date}'
;

-- FileName:   /data/output/einv/${batch_date}/cur/cur_einvoice_account_f.${batch_date}.dat

"""

print(sql_to_pyspark_df(sql_query))

df = (
    spark.table(f"""{params["cur_schema"]}.einvoice_account""")
    .filter(col("etl_dt") == params["batch_date"])
    .select(
        regexp_replace(regexp_replace(col("account_id"), (13), ""), (10), "").alias("account_id")
        ,regexp_replace(regexp_replace(col("customer_id"), (13), ""), (10), "").alias("customer_id")
        ,regexp_replace(regexp_replace(col("agent_id"), (13), ""), (10), "").alias("agent_id")
        ,regexp_replace(regexp_replace(col("status_code"), (13), ""), (10), "").alias("status_code")
        ,regexp_replace(regexp_replace(col("cust_full_name"), (13), ""), (10), "").alias("cust_full_name")
        ,regexp_replace(regexp_replace(col("home_branch"), (13), ""), (10), "").alias("home_branch")
        ,regexp_replace(regexp_replace(col("supplier_id"), (13), ""), (10), "").alias("supplier_id")
        ,regexp_replace(regexp_replace(col("source_system_id"), (13), ""), (10), "").alias("source_system_id")
        ,regexp_replace(regexp_replace(col("lob_ty

In [5]:
from jinja.environment import render_template


def run_migration_pipeline(script_name):
    input_file, file_name, input_file = get_paths(script_name)

    # ==========================================
    # BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
    # ==========================================
    decomposer = CurSqlDecomposer(source_rules)
    decomposed_script = decomposer.decompose(input_file)

    # Ghi file sub-SQL ra ổ đĩa
    writer = CurDecomposerWriter()
    writer.write(decomposed_script, output_root / file_name)
    print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")


    # ==========================================
    # BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
    # ==========================================
    processor = CurMetadataProcessor(source_rules)

    try:
        # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
        pipeline_config = processor.process(decomposed_script, input_file)

        # Ghi file YAML
        metadata_output_dir = output_root / file_name / "metadata"
        processor.write_yaml(pipeline_config, metadata_output_dir)

        print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
        print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
        print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
        print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

    except FileNotFoundError as e:
        print(f"❌ [Lỗi Bước 2]: {e}")
        print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


    # print("==========================================")
    # print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
    # print("==========================================")

    with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
        pipeline_config = yaml.safe_load(f)

    generator = CurPySparkGenerator(source_rules, output_mode="simple")
    ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)


    print("🎉 Hoàn tất toàn bộ Pipeline!")


In [6]:

table_list = [
    # "ailab_mhbos_t_trust_acc",
    # "ailab_mhbos_t_ledger_hd",
    # "ailab_mhbos_t_ledger_dt",
    # "ailab_mhbos_t_contract",
    # "ailab_mhbos_m_client_ext",
    # "ailab_mhbos_m_client",
    # "ailab_mhbos_ecmit029r",
    # "ailab_mhbos_ecmit028r",
    # "ailab_dm_sbl_loan_position",
    # "ailab_dm_sbl_loan",
    # "ailab_dm_margin_position",
    # "ailab_dm_log_field_value_change",
    # "ailab_dm_digital_invest_position",
    # "ailab_dm_customer_custom",

    # Từ ảnh 3
    # "ailab_dm_customer",
    # "ailab_dm_cash_movement",
    "ailab_dm_account_cif",
    "ailab_mhbos_m_client"

    # Từ ảnh 4
    # "einvoice_account"
]

for table in table_list:
    try:
        run_migration_pipeline(f"cur_{table}")
    except:
        run_migration_pipeline(f"cur_{table}")



File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\unl\unl_cur_ailab_dm_account_cif.sql
✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\unl_cur_ailab_dm_account_cif\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: []
   -> File YAML đã lưu tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\unl_cur_ailab_dm_account_cif\metadata\unl_cur_ailab_dm_account_cif.yaml
Error rendering DDL template model_1/cur_ddl.jinja: 'NoneType' object has no attribute 'lower'
Error rendering DDL template model_2a/cur_ddl.jinja: 'NoneType' object has no attribute 'lower'
Error rendering DDL template model_2b/cur_ddl.jinja: 'NoneType' object has no attribute 'lower'
Error rendering DDL template model_3/cur_ddl.jinja: 'NoneType' object has no attribute 'lower'
Error rendering DDL template model_3/cur_h_ddl.jinja: 'NoneType' object has no a

In [7]:
import os

def remove_file_prefix(folder_path: str, prefix: str = "cur_cur_unl_"):
    """
    Scans the directory and strips the specified prefix from all filenames.
    """
    if not os.path.exists(folder_path):
        print(f"Error: The path '{folder_path}' does not exist.")
        return

    renamed_count = 0

    # os.scandir is significantly faster than os.listdir for large numbers of files
    with os.scandir(folder_path) as entries:
        for entry in entries:
            # Only process files, skip directories
            if entry.is_file() and entry.name.startswith(prefix):
                old_path = entry.path
                # Strip the prefix from the beginning of the filename
                new_name = entry.name[len(prefix):]
                new_path = os.path.join(folder_path, new_name)

                try:
                    os.rename(old_path, new_path)
                    renamed_count += 1
                except OSError as e:
                    print(f"Failed to rename {entry.name}: {e}")

    print(f"Success! Processed and removed prefix from {renamed_count} files.")

# Execution block
if __name__ == "__main__":
    TARGET_DIR = r"C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\unload"
    remove_file_prefix(TARGET_DIR)

Failed to rename cur_cur_unl_cur_ailab_mhbos_m_client.py: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\ext_giadung\\projects\\hql_spark_bridge\\output\\migration\\unload\\cur_cur_unl_cur_ailab_mhbos_m_client.py' -> 'C:\\Users\\ext_giadung\\projects\\hql_spark_bridge\\output\\migration\\unload\\cur_ailab_mhbos_m_client.py'
Success! Processed and removed prefix from 1 files.


In [8]:
from pathlib import Path

def merge_ddl_to_single_file(table_list: list, output_merged_path: str):
    """
    Reads DDL scripts for a list of tables and merges them into a single file.
    Ensures no table DDL is duplicated in the output.
    """
    output_file = Path(output_merged_path)
    # Ensure the parent directory for the merged file exists
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Base directory where individual DDL files are located
    base_ddl_dir = Path("C:/Users/ext_giadung/projects/hql_spark_bridge/output/migration/ddl/cur")

    # Use a set to keep track of processed tables and avoid duplicates
    processed_tables = set()

    # Open the merged file once in write mode to clear old content, then append
    with open(output_file, "w", encoding="utf-8") as merged_f:
        merged_f.write(f"-- ─────────────────────────────────────────────────────────\n")
        merged_f.write(f"-- MERGED DDL MIGRATION SCRIPT\n")
        merged_f.write(f"-- ─────────────────────────────────────────────────────────\n\n")

        for table in table_list:
            table_lower = table.lower().strip()

            # Skip if the table has already been processed in this run
            if table_lower in processed_tables:
                print(f"Table '{table_lower}' already processed. Skipping duplicate.")
                continue

            ddl_content = ""
            current_target_table = table_lower

            # 1. Try reading the existing DDL file first
            expected_path = base_ddl_dir / f"cur_{table_lower}.sql"

            try:
                ddl_content = expected_path.read_text(encoding="utf-8")
                print(f"Found existing DDL for: {table_lower}")

            except FileNotFoundError:
                # 2. Fallback: Decompose and generate DDL if file doesn't exist
                print(f"DDL file not found for '{table_lower}'. Triggering decomposer...")
                try:
                    # Fixed the double input_file assignment bug
                    input_file, file_name, _ = get_paths(f"cur_{table_lower}")

                    decomposer = CurSqlDecomposer(source_rules)
                    decomposed_script = decomposer.decompose(input_file)

                    current_target_table = decomposed_script.target_table.lower()

                    # Double-check if the decomposed target table was already processed
                    if current_target_table in processed_tables:
                        print(f"Decomposed target table '{current_target_table}' already processed. Skipping.")
                        continue

                    generated_path = base_ddl_dir / f"cur_{current_target_table}.sql"
                    ddl_content = generated_path.read_text(encoding="utf-8")

                except Exception as e:
                    print(f"[Error] Failed to decompose or read DDL for table {table_lower}: {e}")
                    continue

            # 3. Write to the merged file if valid content was retrieved
            if ddl_content.strip():
                merged_f.write(f"-- Block: cur_{current_target_table}\n")
                merged_f.write(ddl_content.strip())
                merged_f.write("\n\n-- ─────────────────────────────────────────────────────────\n\n")

                # Mark this table as completed to lock it from future duplicate writes
                processed_tables.add(current_target_table)
                if current_target_table != table_lower:
                    processed_tables.add(table_lower)

    print(f"\nSuccessfully merged {len(processed_tables)} unique table DDLs into: {output_file}")

In [9]:
merge_ddl_to_single_file(table_list, r"C:\Users\ext_giadung\projects\hql_spark_bridge\docs\reference\testing\cur_ddl.sql")

DDL file not found for 'ailab_dm_account_cif'. Triggering decomposer...
File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\unl\unl_cur_ailab_dm_account_cif.sql
[Error] Failed to decompose or read DDL for table ailab_dm_account_cif: 'NoneType' object has no attribute 'lower'
DDL file not found for 'ailab_mhbos_m_client'. Triggering decomposer...
File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\unl\unl_cur_ailab_mhbos_m_client.sql
[Error] Failed to decompose or read DDL for table ailab_mhbos_m_client: 'NoneType' object has no attribute 'lower'

Successfully merged 0 unique table DDLs into: C:\Users\ext_giadung\projects\hql_spark_bridge\docs\reference\testing\cur_ddl.sql


In [10]:
template_dir = PROJECT_ROOT / "template" / "migration"
model_folders = [d for d in template_dir.iterdir() if d.is_dir() and d.name.startswith('model_')]

for model_folder in model_folders:
    model_name = model_folder.name
    print(model_name.split("_")[-1].lower())
    # print(f"Processing model: {model_name}")
    # enricher = DdlResolver(source_rules=source_rules)
    # ddl_context = enricher.enrich(pipeline_config, model_type=model_name.split()[-1].lower())

1
2a
2b
3
3a
3b
4
5a
5b
6
